In [ ]:
# === CONFIGURE YOUR TRADE HERE ===
GIVE = ["Player Name A", "Player Name B"]  # players you are giving away
RECEIVE = ["Player Name C"]               # players you are receiving
GAMES_REMAINING = 4                        # games left in current week

In [ ]:
import sys; sys.path.insert(0, "..")
from fantasy.yahoo_client import get_my_roster, get_current_matchup, get_roster_by_team
from fantasy.nba_stats import get_player_stats, get_games_this_week
from fantasy.analysis import project_team_categories, classify_categories, trade_category_delta
from fantasy.llm import build_trade_prompt, ask_gemini
import pandas as pd

In [ ]:
give_stats = []
for name in GIVE:
    stats = get_player_stats(name)
    if stats is None:
        print(f"Warning: no stats found for {name}")
    else:
        give_stats.append(stats)

receive_stats = []
for name in RECEIVE:
    stats = get_player_stats(name)
    if stats is None:
        print(f"Warning: no stats found for {name}")
    else:
        receive_stats.append(stats)

In [ ]:
matchup = get_current_matchup()
week_start, week_end = matchup["week_start"], matchup["week_end"]
my_roster = get_my_roster()
opp_roster = get_roster_by_team(matchup["opponent_team_key"])

def enrich(roster):
    return [{**p, "stats": get_player_stats(p["name"]),
             "games_remaining": get_games_this_week(p["team_abbr"], week_start, week_end)}
            for p in roster]

my_players = enrich(my_roster)
opp_players = enrich(opp_roster)
my_proj = project_team_categories(my_players)
opp_proj = project_team_categories(opp_players)
matchup_context = classify_categories(my_proj, opp_proj)

In [ ]:
delta = trade_category_delta(my_players, give_stats, receive_stats, GAMES_REMAINING)

df = pd.DataFrame({
    "Delta": delta,
    "Matchup Status": matchup_context,
}).round(2)
df["Delta"] = df["Delta"].apply(lambda x: f"{x:+.1f}")
print(f"Trade: Give {GIVE} → Receive {RECEIVE}\n")
print(df.to_string())

In [ ]:
prompt = build_trade_prompt(delta, matchup_context)
advice = ask_gemini(prompt)
print("\n=== TRADE VERDICT ===\n")
print(advice)